Reference : https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html

In [ ]:
# Import libraries
import joblib
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Read the data and re-introduce the train-test split
df = pd.read_csv("../data/processed/nhanes_data_processed_label_encoded.csv")

with open("../data/results/split_data.pkl", "rb") as f:
    split_data = pickle.load(f)

with open("../data/results/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)
feature_names = metadata["feature_names"]

RANDOM_STATE = 42
X_train = split_data["X_train"]
X_train_scaled = split_data["X_train_scaled"]
y_train = split_data["y_train"]
X_test = split_data["X_test"]
X_test_scaled = split_data["X_test_scaled"]
y_test = split_data["y_test"]

In [ ]:
# Load models
dt_model = joblib.load('../models/decision_tree_model.pkl')
rf_model = joblib.load('../models/random_forest_model.pkl')
xgb_model = joblib.load('../models/xgboost_model.pkl')
adb_model = joblib.load('../models/adaboost_model.pkl')
lgr_model = joblib.load('../models/logistic_regression_model.pkl')
knn_model = joblib.load('../models/k-nearest_neighbors_model.pkl')
svc_model = joblib.load('../models/support_vector_machine_model.pkl')
nn_model = tf.keras.models.load_model("../models/neural_network_model.keras", compile=True)

In [ ]:
# Build stacking ensemble (excluding neural network for scikit-learn StackingClassifier)
base_models = [
    ("rf", rf_model),
    ("xgb", xgb_model),
    ("lgr", lgr_model)
    ]

# Define meta-model
meta_model = LogisticRegression(class_weight="balanced", max_iter=100, multi_class="multinomial", n_jobs=-1, random_state=RANDOM_STATE)

# Create StratifiedKFold for cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Create and fit the stacking ensemble
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=cv,
    n_jobs=-1,
    stack_method="predict_proba",
    passthrough=False
    )

# Define the parameter grid for optimising the meta-model
param_grid = {"final_estimator__penalty" : ["l2", "l1", "elasticnet"],
                "final_estimator__solver" : ["lbfgs", "newton-cholesky", "saga"],
                "final_estimator__C" : [0.01, 0.1, 1],
                "final_estimator__max_iter": [100, 200]
}

# Initialise and run GridSearchCV
grid_search = GridSearchCV(estimator=stacking_clf,
                            param_grid=param_grid,
                            scoring="f1_macro",
                            cv=cv,
                            n_jobs=-1,
                            verbose=2
                            )

# Build the grid search using the training set
grid_search.fit(X_train, y_train)

print(f"Best hyperparameters values: {grid_search.best_params_}")

In [ ]:
# Create the optimised meta-model and stacking ensemble
meta_model_optimised = LogisticRegression(class_weight="balanced", C=1, max_iter=100, penalty="l2", multi_class="multinomial", solver="lbfgs", n_jobs=-1, random_state=RANDOM_STATE)

stacking_clf_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model_optimised,
    cv=cv,
    n_jobs=-1,
    stack_method="predict_proba",
    passthrough=False
    )

# Fit the stacking ensemble with training
stacking_clf_model.fit(X_train, y_train)

In [ ]:
# Predict using the stacking ensemble|
y_pred = stacking_clf_model.predict(X_test)
y_pred

In [ ]:
# Generate confusion matrix and display with each row and column labelled
sclf_cnf_matrix_test = confusion_matrix(y_test, y_pred, labels=stacking_clf_model.classes_)

cnf_matrix_test_df = pd.DataFrame(sclf_cnf_matrix_test, columns=["Diabetes_yes_pred", "Diabetes_no_pred", "Diabetes_borderline_pred"], 
                                    index=["Diabetes_yes_actual", "Diabetes_no_actual", "Diabetes_borderline_actual"])
cnf_matrix_test_df

In [ ]:
# Generate classification report
sclf_model_classfication_report = classification_report(y_test, y_pred, target_names=["Diabetes_yes", "Diabetes_no", "Diabetes_borderline"], output_dict=True)
pd.DataFrame.from_dict(sclf_model_classfication_report).T

In [ ]:
# Generate the ROC AUC score
from sklearn.metrics import roc_auc_score

y_pred_proba_sclf = stacking_clf_model.predict_proba(X_test)

sclf_model_roc_auc = roc_auc_score(y_test, y_pred_proba_sclf, average="macro", multi_class="ovr")
print(sclf_model_roc_auc)

## Results comparison

In [ ]:
# Import libraries
import pandas as pd
import pickle

In [ ]:
# Load results
with open("../data/results/model_results.pkl", "rb") as f:
    results = pickle.load(f)

confusion_matrices = results["confusion_matrices"]
auc_scores = results["auc_scores"]
classification_reports = results["classification_reports"]
pred_proba = results["pred_proba"]

In [ ]:
confusion_matrices["Stacked Ensemble"] = sclf_cnf_matrix_test
auc_scores["Stacked Ensemble"] = sclf_model_roc_auc
classification_reports["Stacked Ensemble"] = sclf_model_classfication_report
pred_proba["Stacked Ensemble"] = y_pred_proba_sclf

In [ ]:
# Iterate through the models and combine the macro precision, recall, f1-score, and ROC AUC into a single DataFrame
results_comparison = {}

for model_name in classification_reports.keys():
    classification_report = classification_reports[model_name]
    auc_score = auc_scores[model_name]
    
    results_comparison[model_name] = {
        "Macro Precision" : classification_report["macro avg"]["precision"],
        "Macro Recall" : classification_report["macro avg"]["recall"],
        "Macro F1" : classification_report["macro avg"]["f1-score"],
        "ROC AUC" : auc_scores[model_name]
    }

results_comparison_df = pd.DataFrame.from_dict(results_comparison, orient="index")
results_comparison_df = results_comparison_df.round(4).sort_values(["ROC AUC", "Macro F1", "Macro Recall", "Macro Precision"], ascending=False)

print(f"Model results comparison:\n{results_comparison_df}")

In [ ]:
# Iterate through the models and compare the recall scores of "Diabetes_yes"
yes_recall_comparison = {}

for model_name, classification_report in classification_reports.items():
    
    yes_recall_comparison[model_name] = {
        "Recall" : classification_report["Diabetes_yes"]["recall"]
    }
    
    yes_recall_comparison_df = pd.DataFrame.from_dict(yes_recall_comparison, orient="index")
    
yes_recall_comparison_df = yes_recall_comparison_df.round(4).sort_values("Recall", ascending=False)

print(f"[Diabetes_yes] Recall comparison:\n{yes_recall_comparison_df}")

In [ ]:
# Iterate through the models and compare the precision scores of "Diabetes_no"
no_precision_f1_comparison = {}

for model_name, classification_report in classification_reports.items():
    
    no_precision_f1_comparison[model_name] = {
        "Precision" : classification_report["Diabetes_no"]["precision"],
        "F1" : classification_report["Diabetes_no"]["f1-score"]
    }
    
    no_precision_comparison_df = pd.DataFrame.from_dict(no_precision_f1_comparison, orient="index")
    
no_precision_comparison_df = no_precision_comparison_df.round(4).sort_values("F1", ascending=False)

print(f"[Diabetes_no] Precision and F1 comparison:\n{no_precision_comparison_df}")

In [ ]:
# Iterate through the models and compare the recall scores of "Diabetes_borderline"
borderline_recall_comparison = {}

for model_name, classification_report in classification_reports.items():
    
    borderline_recall_comparison[model_name] = {
        "Recall" : classification_report["Diabetes_borderline"]["recall"]
    }
    
    borderline_recall_comparison_df = pd.DataFrame.from_dict(borderline_recall_comparison, orient="index")
    
borderline_recall_comparison_df = borderline_recall_comparison_df.round(4).sort_values("Recall", ascending=False)

print(f"[Diabetes_borderline] Recall comparison:\n{borderline_recall_comparison_df}")